# 03 — Darmstadt DOP20: Zero-Shot + TTPA Inference & OSM Evaluation

**Purpose:** run the trained RG-GeoPrompt model on Darmstadt DOP20 patches,
with and without TTPA, detect prediction collapse, evaluate against eroded
OSM pseudo-GT, and produce the 5-column report figures.

**Needs:** Kaggle **T4 GPU**, `darmstadt-dop20` patch dataset attached,
trained `geoprompt_best.pth` (local or HF), `HF_TOKEN` secret.

**Outputs:** prediction masks (`darmstadt_preds/`), figures
(`darmstadt_figs/`), `osm_pseudo_gt_f1.csv` — all in `/kaggle/working`.

**Evaluation is patch-based by design — predictions are NEVER stitched
into full tiles** (avoids seam artifacts; handoff §8).

> SOURCE: handoff spec — these cells have NOT yet run on Kaggle
> (DOP20 not downloaded at time of writing). Treat as reviewed drafts.

In [ ]:
# Bootstrap: clone repo and add src/ to sys.path. Run every session.
import subprocess, sys
from pathlib import Path

if Path("/kaggle/input").exists():                 # Kaggle
    CLONE_DIR = Path("/kaggle/working/VC/rg-geoprompt-peft")
    if not CLONE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/HarishDeepak/VC.git", str(CLONE_DIR)],
            check=True)
        print(f"✓ cloned → {CLONE_DIR}")
    else:
        print(f"✓ repo present → {CLONE_DIR}")
    sys.path.insert(0, str(CLONE_DIR / "src"))
else:                                              # local (VS Code)
    for cand in ["../src", "src"]:
        if Path(cand, "rg_geoprompt").exists():
            sys.path.insert(0, str(Path(cand).resolve()))
            break

from rg_geoprompt import paths
print(paths.describe())

In [ ]:
# Installs (idempotent, ~1 min). Run every session.
%pip install -q peft open_clip_torch
print("✓ peft + open_clip_torch installed")

## Step 0 (LOCAL machine, one-time) — preprocess DOP20

Download 4–6 tiles for central Darmstadt from
**gds.hessen.de → Luftbildinformationen → DOP20** (free, no registration).
Tiles are 1km×1km GeoTIFF, EPSG:25832, **4-band RGBI** — the slicer keeps
only RGB and writes a `transforms.json` so OSM rasterization works on
Kaggle. Then upload the output folder as Kaggle dataset `darmstadt-dop20`.

In [ ]:
# RUN LOCALLY (needs rasterio + the downloaded .jpg/.jgw files). Skip on Kaggle.
RUN_LOCAL_PREPROCESS = False
if RUN_LOCAL_PREPROCESS:
    from pathlib import Path
    from rg_geoprompt.datasets import slice_dop20_tile
    tif_dir = Path("data/dop20_raw")          # downloaded Hessen DOP20 (JPG+JGW)
    out_img = Path("data/darmstadt_dop20/images")
    out_ndvi = Path("data/darmstadt_dop20/ndvi")
    total = 0
    for tif in sorted(tif_dir.glob("*.jpg")):  # Hessen DOP20: JPG+JGW pairs
        n = slice_dop20_tile(tif, out_img, out_ndvi)
        total += n
        print(f"{tif.name}: {n} patches")
    print(f"total: {total} — upload data/darmstadt_dop20 as 'darmstadt-dop20'")

## Load Darmstadt patches + trained GeoPrompt model

In [ ]:
# REQUIRES: Kaggle T4 GPU
import torch
from torch.utils.data import DataLoader
from rg_geoprompt import paths
from rg_geoprompt.datasets import DarmstadtPatchDataset
from rg_geoprompt.models_geoprompt import load_geoprompt
from rg_geoprompt.prompts import load_or_encode_text_embeddings
from rg_geoprompt.utils import ensure_checkpoint

text_embeddings = load_or_encode_text_embeddings()
# lowres_similarity=True matches how geoprompt_best.pth was trained (T4 OOM fallback)
geo_model = load_geoprompt(ensure_checkpoint("geoprompt_best.pth"),
                           text_embeddings, lowres_similarity=True)
geo_model.eval()

darm_ds = DarmstadtPatchDataset()
darm_loader = DataLoader(darm_ds, batch_size=4, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"Darmstadt patches: {len(darm_ds)}")

In [ ]:
# Activate Darmstadt mode: max-pool prompt ensemble + orthogonal projection.
# This swaps in red-tile-aware building prompts WITHOUT touching the trained
# backbone or text_proj — inference-only change, no retraining needed.
from rg_geoprompt.prompts import encode_per_prompt, DARMSTADT_PROMPTS

multi_emb = encode_per_prompt(DARMSTADT_PROMPTS)   # [6, 8, 512]
geo_model.set_darmstadt_mode(multi_emb, ortho_scale=0.3)
print(f"Darmstadt mode active | prompt shape: {tuple(multi_emb.shape)}")
print("  building: 8 prompts (red terracotta + gray flat) via max-pool")
print("  clutter : 5 prompts (ground-level only, no red-surface language)")
print(f"  ortho_scale=0.3 → building score -= 0.3 × clutter score")

# REQUIRES: Kaggle T4 GPU
# SOURCE: handoff spec — not yet run on Kaggle
import numpy as np
from rg_geoprompt.constants import DEVICE
from rg_geoprompt.ttpa import detect_collapse, entropy_map, ttpa_predict

pred_dir = paths.WORK_DIR / "darmstadt_preds"
pred_dir.mkdir(exist_ok=True)

zs_preds, ttpa_preds, entropies, collapse_flags = {}, {}, {}, []
for imgs, stems in darm_loader:
    imgs = imgs.to(DEVICE)
    # Use params chosen by the diagnostic cell above
    final_logits, ref_probs = ttpa_predict(
        geo_model, imgs,
        n_steps=TTPA_N_STEPS, lr=TTPA_LR_USE, kl_weight=TTPA_KL_USE,
        masked_entropy=True,   # only uncertain pixels drive entropy loss
    )
    stats = detect_collapse(final_logits)
    collapse_flags.append(stats["collapsed"])
    ent = entropy_map(final_logits).cpu().numpy()
    zs = ref_probs.argmax(dim=1).cpu().numpy()
    tt = final_logits.argmax(dim=1).cpu().numpy()
    for i, stem in enumerate(stems):
        zs_preds[stem], ttpa_preds[stem], entropies[stem] = zs[i], tt[i], ent[i]
        np.save(pred_dir / f"{stem}_ttpa.npy", tt[i].astype(np.uint8))
        np.save(pred_dir / f"{stem}_zs.npy", zs[i].astype(np.uint8))

frac = sum(collapse_flags) / max(len(collapse_flags), 1)
print(f"batches flagged as collapsed: {100*frac:.1f}%")
if frac > 0.1:
    print("⚠ COLLAPSE — rerun ttpa_predict with n_steps=1, lr=5e-6, kl_weight=1.0")

## TTPA diagnostic — run before full inference
Measures weight delta and prediction pixel change to determine which TTPA
failure mode is active before committing to full inference settings.

In [ ]:
# REQUIRES: Kaggle T4 GPU
# Run on ONE batch before full inference to choose TTPA settings.
# Interpet output:
#   weight_delta < 1e-5          → gradient not flowing → check requires_grad
#   weight_delta > 0, change < 1% → L2-norm cancelling → increase lr to 1e-4
#   change > 5%                  → TTPA working, use these params for full run
from rg_geoprompt.constants import (TTPA_DARMSTADT_STEPS, TTPA_DARMSTADT_LR,
                                     TTPA_DARMSTADT_KL_WEIGHT)
from rg_geoprompt.ttpa import ttpa_diagnostic

diag_imgs, _ = next(iter(darm_loader))
diag_imgs = diag_imgs.to(DEVICE)

diag = ttpa_diagnostic(geo_model, diag_imgs,
                        n_steps=TTPA_DARMSTADT_STEPS,
                        lr=TTPA_DARMSTADT_LR,
                        kl_weight=TTPA_DARMSTADT_KL_WEIGHT)

# Based on output, set these for the inference cell below:
# Standard (handoff):  n_steps=2,  lr=1e-5,  kl_weight=0.5
# Darmstadt-tuned:     n_steps=5,  lr=5e-5,  kl_weight=0.1  (if change < 1%)
# Aggressive:          n_steps=5,  lr=1e-4,  kl_weight=0.05 (if L2-norm issue)
TTPA_N_STEPS   = TTPA_DARMSTADT_STEPS
TTPA_LR_USE    = TTPA_DARMSTADT_LR
TTPA_KL_USE    = TTPA_DARMSTADT_KL_WEIGHT
print(f"\nUsing: steps={TTPA_N_STEPS}, lr={TTPA_LR_USE}, kl={TTPA_KL_USE}")

## SAM-encoder → PCA → DenseCRF boundary refinement
Uses SAM-ViT-B image encoder features (no decoder, no AMG) as the bilateral
term in DenseCRF, replacing raw RGB. Avoids over-segmentation and shadow
snapping that plague AMG + majority-voting on aerial imagery.

In [ ]:
# REQUIRES: Kaggle T4 GPU
# Install dependencies (pydensecrf has a C extension — ~2 min first time)
%pip install -q segment-anything pydensecrf scikit-learn
print("✓ SAM + pydensecrf + sklearn installed")

In [ ]:
# REQUIRES: Kaggle T4 GPU
import torch
import numpy as np
from PIL import Image
from rg_geoprompt import paths
from rg_geoprompt.constants import DEVICE
from rg_geoprompt.crf_refine import (ensure_sam_checkpoint, load_sam_encoder,
                                      extract_sam_features, refine_with_sam_crf)

# Download SAM-ViT-B checkpoint (~375 MB, one-time)
sam_ckpt = ensure_sam_checkpoint(paths.WORK_DIR)
sam_encoder = load_sam_encoder(sam_ckpt, device=DEVICE)

# Run CRF refinement on all patches that have TTPA predictions.
# We rerun geo_model in eval mode to get raw logits (not argmax).
# SAM encoder stays loaded throughout — both fit in T4 VRAM concurrently.
crf_preds = {}
geo_model.eval()

for imgs, stems in darm_loader:
    imgs = imgs.to(DEVICE)

    # Raw logits from geo_model (before argmax)
    with torch.no_grad():
        logits = geo_model(imgs)                              # [B, 6, 512, 512]

    for i, stem in enumerate(stems):
        logits_np = logits[i].float().cpu().numpy()           # [6, 512, 512]

        # Reconstruct uint8 RGB from normalised tensor for SAM
        rgb = imgs[i].cpu().float().numpy()                   # [3, 512, 512]
        rgb = (rgb * np.array([0.229, 0.224, 0.225])[:, None, None]
               + np.array([0.485, 0.456, 0.406])[:, None, None])
        rgb = (rgb.clip(0, 1) * 255).astype(np.uint8)
        rgb = rgb.transpose(1, 2, 0)                          # [H, W, 3]

        sam_feats = extract_sam_features(sam_encoder, rgb,
                                         out_size=(512, 512)) # [512, 512, 256]
        crf_pred = refine_with_sam_crf(logits_np, sam_feats)  # [512, 512]
        crf_preds[stem] = crf_pred
        np.save(paths.WORK_DIR / "darmstadt_preds" / f"{stem}_crf.npy",
                crf_pred.astype(np.uint8))

print(f"CRF refinement done: {len(crf_preds)} patches")
print("Inspect a few patches visually before running OSM F1.")

In [ ]:
# REQUIRES: Kaggle T4 GPU
# SOURCE: handoff spec — not yet run on Kaggle
import numpy as np
from rg_geoprompt.constants import DEVICE
from rg_geoprompt.ttpa import detect_collapse, entropy_map, ttpa_predict

pred_dir = paths.WORK_DIR / "darmstadt_preds"
pred_dir.mkdir(exist_ok=True)

zs_preds, ttpa_preds, entropies, collapse_flags = {}, {}, {}, []
for imgs, stems in darm_loader:
    imgs = imgs.to(DEVICE)
    final_logits, ref_probs = ttpa_predict(geo_model, imgs)  # restores text_proj
    stats = detect_collapse(final_logits)
    collapse_flags.append(stats["collapsed"])
    ent = entropy_map(final_logits).cpu().numpy()
    zs = ref_probs.argmax(dim=1).cpu().numpy()
    tt = final_logits.argmax(dim=1).cpu().numpy()
    for i, stem in enumerate(stems):
        zs_preds[stem], ttpa_preds[stem], entropies[stem] = zs[i], tt[i], ent[i]
        np.save(pred_dir / f"{stem}_ttpa.npy", tt[i].astype(np.uint8))
        np.save(pred_dir / f"{stem}_zs.npy", zs[i].astype(np.uint8))

frac = sum(collapse_flags) / max(len(collapse_flags), 1)
print(f"batches flagged as collapsed: {100*frac:.1f}%")
if frac > 0.1:
    print("⚠ COLLAPSE — rerun with ttpa_predict(..., n_steps=1, lr=5e-6, "
          "kl_weight=1.0) and inspect entropy maps (should NOT be uniformly low)")

## OSM pseudo-GT — download, rasterize, erode (3×3)
Reported as **OSM pseudo-GT F1** — OSM is noisy, never call it GT.
Boundary creation differs here (erosion → 255); the 255-exclusion rule
in metrics is identical to Potsdam.

In [ ]:
# SOURCE: handoff spec — not yet run on Kaggle
%pip install -q osmnx
import json
import numpy as np
from affine import Affine
from rg_geoprompt import paths
from rg_geoprompt.osm_eval import (download_osm_darmstadt, erode_osm_labels,
                                   rasterize_osm_patch)

gdf = download_osm_darmstadt()           # EPSG:25832, ~1-2 min
print(f"OSM features: {len(gdf)}")

transforms = json.loads(
    (paths.DARMSTADT_IMG_DIR / "transforms.json").read_text())

osm_masks = {}
for stem in ttpa_preds:                  # only patches we predicted on
    tf = Affine(*transforms[stem])
    raw = rasterize_osm_patch(gdf, tf, shape=(512, 512))
    osm_masks[stem] = erode_osm_labels(raw)   # 3x3 erosion → borders = 255

# Sanity: visually confirm building masks align with actual buildings.
# If offset by >a few pixels → check CRS, then raise erosion kernel to 5.

## OSM pseudo-GT F1 (6 classes, only 255 excluded)

In [ ]:
# SOURCE: handoff spec — not yet run on Kaggle
import csv
from rg_geoprompt.constants import CLASS_NAMES
from rg_geoprompt.osm_eval import osm_pseudo_gt_f1

res_zs = osm_pseudo_gt_f1(zs_preds, osm_masks)
res_tt = osm_pseudo_gt_f1(ttpa_preds, osm_masks)

with open(paths.WORK_DIR / "osm_pseudo_gt_f1.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["class", "zero_shot_f1", "ttpa_f1"])
    for i, name in enumerate(CLASS_NAMES):
        w.writerow([name, f"{res_zs['per_class_f1'][i]:.4f}",
                    f"{res_tt['per_class_f1'][i]:.4f}"])
    w.writerow(["MEAN", f"{res_zs['mean_f1']:.4f}", f"{res_tt['mean_f1']:.4f}"])

print(f"{'class':12s} {'zero-shot':>10s} {'TTPA':>10s}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:12s} {res_zs['per_class_f1'][i]:10.4f} "
          f"{res_tt['per_class_f1'][i]:10.4f}")
print(f"{'MEAN':12s} {res_zs['mean_f1']:10.4f} {res_tt['mean_f1']:10.4f}")
print("(Cars/Clutter have no OSM layer → those rows reflect 255-masked "
      "pixels only; discuss in the report.)")

## 5-column report figures: RGB | zero-shot | TTPA | entropy | OSM

In [ ]:
# SOURCE: handoff spec — not yet run on Kaggle
import random
import numpy as np
from PIL import Image
from rg_geoprompt.utils import five_column_figure

fig_dir = paths.WORK_DIR / "darmstadt_figs"
fig_dir.mkdir(exist_ok=True)

for stem in random.sample(sorted(ttpa_preds), k=min(12, len(ttpa_preds))):
    rgb = np.array(Image.open(paths.DARMSTADT_IMG_DIR / f"{stem}.png"))
    five_column_figure(rgb, zs_preds[stem], ttpa_preds[stem],
                       entropies[stem], osm_masks.get(stem),
                       title=stem, save_path=fig_dir / f"{stem}.png")
print(f"figures saved to {fig_dir}")
# Healthy transfer: entropy LOW inside regions, HIGH only at boundaries.
# Uniformly high entropy = transfer failed. Uniformly low = collapse.

## Backup + checklist

In [ ]:
from rg_geoprompt.utils import hf_backup
hf_backup()
print("✓ Darmstadt outputs backed up")
print("Checklist: preds saved ✓ | figures saved ✓ | osm_pseudo_gt_f1.csv ✓")
print("           collapse check done ✓ | nothing stitched into tiles ✓")